# Response Formatting and Translation Module

**Part of:** Indic Government Scheme Copilot

**Purpose:** Final output layer that:
* Takes scheme + eligibility data from backend modules
* Generates user-friendly responses using Sarvam AI LLM
* Translates to Hindi when needed
* Logs execution with MLflow

**Dependencies:**
* Sarvam AI API (LLM + Translation)
* MLflow for experiment tracking

**Function:** `format_output(schemes, language)`

In [0]:
# Import required libraries
import requests
import mlflow
import json
from typing import List, Dict, Any

# Sarvam AI Configuration
SARVAM_API_KEY = "sk_s3685mrt_I1bEdMa8d7I2H3EQz5hM0Yzj"  # Replace with your actual API key
SARVAM_LLM_URL = "https://api.sarvam.ai/v1/chat/completions"
SARVAM_TRANSLATE_URL = "https://api.sarvam.ai/translate"

# MLflow setup (optional: set tracking URI if using remote MLflow)
# mlflow.set_tracking_uri("databricks")

print("✓ Setup complete")

✓ Setup complete


In [0]:
def generate_response(schemes: List[Dict[str, Any]], language: str = "english") -> str:
    """
    Generate a user-friendly response from scheme data using Sarvam AI LLM.
    
    Args:
        schemes: List of scheme dictionaries with name, description, documents, reason
        language: Target language - "english" or "hindi" (generates directly in target language)
    
    Returns:
        Formatted response string in the target language
    """
    # Prepare scheme information for the LLM
    scheme_text = ""
    for i, scheme in enumerate(schemes, 1):
        scheme_text += f"\n{i}. {scheme['name']}\n"
        scheme_text += f"   Description: {scheme['description']}\n"
        scheme_text += f"   Eligibility: {scheme.get('reason', 'Meets criteria')}\n"
        scheme_text += f"   Documents: {', '.join(scheme.get('documents', []))}\n"
    
    # Adjust prompt based on target language
    if language.lower() == "hindi":
        language_instruction = "Respond in Hindi (Devanagari script). "
    else:
        language_instruction = ""
    
    # Create prompt for Sarvam AI
    prompt = f"""{language_instruction}Explain the following government schemes clearly for a common Indian citizen. Include benefits, eligibility reasons, and required documents. Keep it simple and structured.

Schemes:
{scheme_text}

Format the response with:
- Numbered list for each scheme
- Clear benefit description
- Eligibility reason
- Required documents as bullet points
- Simple, non-technical language
- Keep it concise (under 1500 characters total)
"""
    
    try:
        # Call Sarvam AI LLM
        headers = {
            "Authorization": f"Bearer {SARVAM_API_KEY}",
            "Content-Type": "application/json"
        }
        
        payload = {
            "model": "sarvam-m",
            "messages": [
                {"role": "system", "content": "You are a helpful assistant explaining Indian government schemes in simple language."},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.7,
            "max_tokens": 1000
        }
        
        response = requests.post(SARVAM_LLM_URL, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        
        result = response.json()
        generated_text = result['choices'][0]['message']['content']
        
        print(f"✓ LLM generation successful (direct {language} output)")
        return generated_text.strip()
        
    except requests.exceptions.HTTPError as e:
        print(f"Error calling Sarvam AI LLM: {e}")
        print(f"Status Code: {response.status_code}")
        print(f"Response: {response.text}")
        # Fallback: Return basic formatted text
        fallback = "\n\n".join([
            f"{i}. **{s['name']}**\n   Benefits: {s['description']}\n   Eligibility: {s.get('reason', 'Meets criteria')}\n   Documents: {', '.join(s.get('documents', []))}"
            for i, s in enumerate(schemes, 1)
        ])
        return fallback
    except Exception as e:
        print(f"Unexpected error: {e}")
        # Fallback: Return basic formatted text
        fallback = "\n\n".join([
            f"{i}. **{s['name']}**\n   Benefits: {s['description']}\n   Eligibility: {s.get('reason', 'Meets criteria')}\n   Documents: {', '.join(s.get('documents', []))}"
            for i, s in enumerate(schemes, 1)
        ])
        return fallback

In [0]:
def translate_response(text: str, target_language: str = "hi") -> str:
    """
    Translate text to target language using Sarvam AI Translation.
    
    Args:
        text: English text to translate
        target_language: Target language code (default: 'hi' for Hindi)
    
    Returns:
        Translated text
    """
    try:
        # Translation API uses different auth header
        headers = {
            "api-subscription-key": SARVAM_API_KEY,
            "Content-Type": "application/json"
        }
        
        payload = {
            "input": text,
            "source_language_code": "en-IN",
            "target_language_code": f"{target_language}-IN",
            "speaker_gender": "Male",  # Fixed: Must be 'Male' or 'Female' (capitalized)
            "mode": "formal",
            "model": "mayura:v1",
            "enable_preprocessing": True
        }
        
        response = requests.post(SARVAM_TRANSLATE_URL, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        
        result = response.json()
        translated_text = result.get('translated_text', text)
        
        print("✓ Translation successful")
        return translated_text
        
    except requests.exceptions.HTTPError as e:
        print(f"Error calling Sarvam AI Translation: {e}")
        print(f"Status Code: {response.status_code}")
        print(f"Response: {response.text}")
        print("Returning original text...")
        return text
    except Exception as e:
        print(f"Unexpected translation error: {e}")
        print("Returning original text...")
        return text

In [0]:
def format_output(schemes: List[Dict[str, Any]], language: str = "english") -> str:
    """
    Main function to format scheme data into user-friendly output with optional translation.
    
    Args:
        schemes: List of scheme dictionaries with structure:
                 {
                     "name": str,
                     "description": str,
                     "documents": [str],
                     "reason": str
                 }
        language: "english" or "hindi"
    
    Returns:
        Formatted and optionally translated response string
    """
    # Start MLflow run for logging
    with mlflow.start_run(run_name="response_formatting"):
        # Log parameters
        mlflow.log_param("feature", "response_formatting")
        mlflow.log_param("language", language)
        mlflow.log_param("num_schemes", len(schemes))
        mlflow.log_param("method", "direct_generation")  # New: track method
        
        print(f"Processing {len(schemes)} scheme(s) in {language}...")
        
        # Generate response directly in target language (more efficient!)
        formatted_response = generate_response(schemes, language=language)
        
        # Log additional metrics
        mlflow.log_metric("response_length", len(formatted_response))
        
        print("✓ Response generated successfully")
        
        return formatted_response

In [0]:
# Sample scheme data (mock output from backend modules)
sample_schemes = [
    {
        "name": "Pradhan Mantri Awas Yojana",
        "description": "Housing scheme providing financial assistance for building or buying a house",
        "documents": ["Aadhaar Card", "Income Certificate", "Bank Account Details"],
        "reason": "Eligible because annual income is below ₹6 lakh and applicant does not own a pucca house"
    },
    {
        "name": "Kisan Credit Card",
        "description": "Credit facility for farmers to purchase seeds, fertilizers, and other agricultural inputs",
        "documents": ["Aadhaar Card", "Land Documents", "Bank Passbook"],
        "reason": "Eligible because applicant is a farmer with cultivable land"
    },
    {
        "name": "Ayushman Bharat - PM-JAY",
        "description": "Health insurance scheme providing coverage up to ₹5 lakh per family per year",
        "documents": ["Aadhaar Card", "Ration Card", "Family Income Proof"],
        "reason": "Eligible because family income is below poverty line threshold"
    }
]

# Test 1: English output
print("=" * 60)
print("TEST 1: ENGLISH OUTPUT")
print("=" * 60)
english_output = format_output(sample_schemes, language="english")
print("\n" + english_output)

# Test 2: Hindi output
print("\n\n" + "=" * 60)
print("TEST 2: HINDI OUTPUT")
print("=" * 60)
hindi_output = format_output(sample_schemes, language="hindi")
print("\n" + hindi_output)

TEST 1: ENGLISH OUTPUT
Processing 3 scheme(s) in english...
✓ LLM generation successful (direct english output)
✓ Response generated successfully

<think>
Okay, let's tackle this query. The user wants explanations of three Indian government schemes in simple terms. The user specified that the response should be for a common Indian citizen, so I need to avoid technical jargon.

First, I'll start with the Pradhan Mantri Awas Yojana. The key points are financial assistance for housing, eligibility based on income and not owning a pucca house, and the required documents. I should mention the amount if possible, but since the user didn't provide it, I'll keep it general. Make sure to explain what a pucca house is in simple terms, maybe "permanent house with proper roof and walls."

Next, the Kisan Credit Card. It's about loans for farmers. The benefit is getting credit for agricultural needs. Eligibility is having cultivable land. Documents include land papers and bank details. I should not

In [0]:
# Diagnostic cell to test Sarvam AI endpoints and understand errors
import requests
import json

print("=" * 60)
print("DIAGNOSTIC: Testing Sarvam AI Endpoints (ALL FIXES)")
print("=" * 60)

# Test 1: LLM Endpoint with CORRECTED model
print("\n1. Testing LLM Endpoint...")
print(f"   URL: {SARVAM_LLM_URL}")
print(f"   Model: sarvam-m ✓")

llm_payload = {
    "model": "sarvam-m",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Say hello in one sentence"}
    ],
    "temperature": 0.7,
    "max_tokens": 100
}

headers = {
    "Authorization": f"Bearer {SARVAM_API_KEY}",
    "Content-Type": "application/json"
}

try:
    response = requests.post(SARVAM_LLM_URL, headers=headers, json=llm_payload, timeout=10)
    print(f"   Status: {response.status_code}")
    
    if response.status_code == 200:
        result = response.json()
        content = result.get('choices', [{}])[0].get('message', {}).get('content', 'N/A')
        # Show first 100 chars
        print(f"   ✓ LLM API works!")
        print(f"   Response preview: {content[:100]}...")
    else:
        print(f"   ✗ Failed: {response.text}")
except Exception as e:
    print(f"   ✗ Exception: {e}")

# Test 2: Translation Endpoint with ALL CORRECTIONS
print("\n2. Testing Translation Endpoint...")
print(f"   URL: {SARVAM_TRANSLATE_URL}")
print(f"   Auth: api-subscription-key ✓")
print(f"   Speaker Gender: Male ✓")

translate_payload = {
    "input": "Hello world",
    "source_language_code": "en-IN",
    "target_language_code": "hi-IN",
    "speaker_gender": "Male",  # FIXED: Capitalized Male/Female
    "mode": "formal",
    "model": "mayura:v1",
    "enable_preprocessing": True
}

translate_headers = {
    "api-subscription-key": SARVAM_API_KEY,
    "Content-Type": "application/json"
}

try:
    response = requests.post(SARVAM_TRANSLATE_URL, headers=translate_headers, json=translate_payload, timeout=10)
    print(f"   Status: {response.status_code}")
    
    if response.status_code == 200:
        result = response.json()
        print(f"   ✓ Translation API works!")
        print(f"   Translated: {result.get('translated_text', 'N/A')}")
    else:
        print(f"   ✗ Failed: {response.text}")
except Exception as e:
    print(f"   ✗ Exception: {e}")

print("\n" + "=" * 60)
if response.status_code == 200:
    print("✓ ALL SYSTEMS OPERATIONAL!")
else:
    print("Check errors above for remaining issues.")
print("=" * 60)

DIAGNOSTIC: Testing Sarvam AI Endpoints (ALL FIXES)

1. Testing LLM Endpoint...
   URL: https://api.sarvam.ai/v1/chat/completions
   Model: sarvam-m ✓
   Status: 200
   ✓ LLM API works!
   Response preview: <think>
Okay, the user wants me to say hello in one sentence. Let me think about how to do that in a...

2. Testing Translation Endpoint...
   URL: https://api.sarvam.ai/translate
   Auth: api-subscription-key ✓
   Speaker Gender: Male ✓
   Status: 200
   ✓ Translation API works!
   Translated: नमस्कार

✓ ALL SYSTEMS OPERATIONAL!


## Production Integration Example

### How to Use This Module

This module is the **final layer** in your government scheme assistant pipeline:

```
User Query → search_schemes() → check_eligibility() → format_output() → Final Response
```

### Example Integration

```python
# Simulated output from other modules
def search_schemes(query):
    # Backend 1 implementation
    return [
        {"name": "PM Kisan", "description": "...", "documents": [...]},
        # ... more schemes
    ]

def check_eligibility(user_profile, schemes):
    # Backend 2 implementation
    # Adds 'reason' field to each scheme
    for scheme in schemes:
        scheme['reason'] = "Eligible because..."
    return schemes

# Full pipeline
user_query = "housing schemes for low income"
user_profile = {"income": 500000, "category": "EWS"}
language = "hindi"

# Step 1: Search for relevant schemes
schemes = search_schemes(user_query)

# Step 2: Check eligibility
eligible_schemes = check_eligibility(user_profile, schemes)

# Step 3: Format output (this module)
final_response = format_output(eligible_schemes, language=language)

# Step 4: Return to user
print(final_response)
```

### Key Features

* **Language Support:** English and Hindi (direct generation)
* **MLflow Tracking:** All operations logged automatically
* **Error Resilience:** Fallback to basic formatting if API fails
* **Scalable:** Easy to add more languages (Tamil, Telugu, Bengali)

### Performance Optimizations

* **Direct multilingual generation** (no translation API needed)
* **Single LLM call** per request
* **Response caching** possible via MLflow
* **Character limit management** (stays under 1500 chars)

### Next Steps for Production

1. **Add More Languages:**
   * Modify `generate_response()` to support: `ta` (Tamil), `te` (Telugu), `bn` (Bengali)
   * Add language instruction in prompt

2. **Add Caching:**
   * Cache common scheme combinations
   * Reduce API costs by 60-80%

3. **Add Monitoring:**
   * Track response times
   * Alert on API failures
   * Monitor MLflow experiments

4. **Add A/B Testing:**
   * Test different prompt templates
   * Compare direct generation vs translation
   * Optimize for user satisfaction

In [0]:
# Quick test showing how other modules would call this
print("✓ Module ready for integration!")
print("\nAvailable function: format_output(schemes, language)")
print("\nSupported languages: 'english', 'hindi'")
print("\nExample call:")
print('  result = format_output(eligible_schemes, language="hindi")')
print("\nMLflow tracking: Enabled ✓")
print("Error handling: Graceful fallbacks ✓")
print("Direct multilingual: Yes ✓")

✓ Module ready for integration!

Available function: format_output(schemes, language)

Supported languages: 'english', 'hindi'

Example call:
  result = format_output(eligible_schemes, language="hindi")

MLflow tracking: Enabled ✓
Error handling: Graceful fallbacks ✓
Direct multilingual: Yes ✓


## Usage Instructions

### Setup Steps
1. **Get Sarvam AI API Key:**
   * Sign up at [Sarvam AI](https://www.sarvam.ai/)
   * Get your API key from the dashboard
   * Replace `your_sarvam_api_key_here` in Cell 2

2. **Install Dependencies:**
   ```
   %pip install requests mlflow
   ```

3. **Run Cells in Order:**
   * Cell 2: Setup and Configuration
   * Cell 3: Helper Function - Generate Response
   * Cell 4: Helper Function - Translate Response
   * Cell 5: Main Function - Format Output
   * Cell 6: Test with Sample Data

### Integration with Other Modules

This module receives input from:
* **Backend 1:** `search_schemes(query)` - returns matching schemes
* **Backend 2:** `check_eligibility(user)` - adds eligibility reasons

Example integration:
```python
# From other modules
schemes = search_schemes("housing")
eligible_schemes = check_eligibility(user_profile, schemes)

# Use this module
final_output = format_output(eligible_schemes, language="hindi")
print(final_output)
```

### Key Features
* **LLM Enhancement:** Uses Sarvam AI to generate natural, user-friendly explanations
* **Translation:** Automatic Hindi translation with cultural context
* **MLflow Tracking:** Logs all operations for monitoring
* **Error Handling:** Graceful fallback if API calls fail
* **Mock Data:** Includes test cases for demo

### Next Steps
* Replace API key with your actual Sarvam AI key
* Run test cells to verify functionality
* Integrate with search and eligibility modules
* Add more language support (Tamil, Telugu, Bengali, etc.)
* Customize response format based on user feedback